# Global Slice PRB Load: 3 gNBs, 5 Fixed UEs

This notebook starts a small `MultiGNBWrapper` environment with three gNodeBs and five fixed UEs across `eMBB`, `URLLC`, and `mMTC` slices. It then plots the upper/global-agent load definition:

`L_{i,s} = PRB_used_{i,s} / PRB_total_{i,s}`

Use the project virtual environment as the notebook kernel.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "multi_gnb_wrapper.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from scenario_creator import create_multignb_env

np.set_printoptions(precision=3, suppress=True)
print(f"Project root: {PROJECT_ROOT}")

## Create The 3-gNB Environment

Scenario index `4` contains eMBB, URLLC, and mMTC L1 slices. The UE positions are fixed (`vx = vy = 0`).

In [ ]:
rng = np.random.default_rng(7)

gnb_configs = [
    {"id": 0, "x": 0.0, "y": 0.0, "coverage_radius": 500.0, "carrier_id": 0, "n_prbs": 100},
    {"id": 1, "x": 450.0, "y": 0.0, "coverage_radius": 500.0, "carrier_id": 0, "n_prbs": 100},
    {"id": 2, "x": 225.0, "y": 390.0, "coverage_radius": 500.0, "carrier_id": 0, "n_prbs": 100},
]

env = create_multignb_env(
    rng=rng,
    n=4,
    gnb_configs=gnb_configs,
    slots_per_step=5,
    L1_level=False,
    step_dt=1e-3,
    mobility_dt=0.0,
    radio_substeps=1,
    max_episode_steps=10,
    use_sumo_mobility=False,
)

# Make slice budgets explicit and easy to read in the load plot.
slice_budgets = {"eMBB": 50, "URLLC": 30, "mMTC": 20}
for gnb in env.gnbs:
    for l1 in gnb.slices_l1:
        slice_type = env.normalize_slice_type(getattr(l1, "type", type(l1).__name__))
        if slice_type in slice_budgets:
            l1.n_prbs = slice_budgets[slice_type]

[(int(gnb.id), [(env.normalize_slice_type(getattr(l1, "type", "")), int(l1.n_prbs)) for l1 in gnb.slices_l1]) for gnb in env.gnbs]

## Add Five Fixed UEs

The PRB allocations below are set manually for a clean global-state demo. This makes the plotted load exactly the HRL global load, independent of scheduler randomness.

In [ ]:
ue_plan = [
    {"gnb_id": 0, "slice_type": "eMBB", "x": 20.0, "y": 0.0, "prbs": 10},
    {"gnb_id": 0, "slice_type": "URLLC", "x": -30.0, "y": 40.0, "prbs": 8},
    {"gnb_id": 1, "slice_type": "mMTC", "x": 460.0, "y": 20.0, "prbs": 5},
    {"gnb_id": 1, "slice_type": "eMBB", "x": 420.0, "y": -30.0, "prbs": 35},
    {"gnb_id": 2, "slice_type": "URLLC", "x": 230.0, "y": 380.0, "prbs": 20},
]

env.clear_ues(reset_ids=True)

for spec in ue_plan:
    ue_id = env.add_ue(
        x=spec["x"],
        y=spec["y"],
        vx=0.0,
        vy=0.0,
        slice_type=spec["slice_type"],
    )
    ue = env.get_ue(ue_id)

    for gnb in env.gnbs:
        gnb.detach_ue(ue_id)

    target_gnb = env._get_gnb_by_id(spec["gnb_id"])
    if spec["slice_type"] != "mMTC":
        target_gnb.attach_ue(ue)

    ue.serving_gnb = int(spec["gnb_id"])
    ue.connected = True
    ue.prbs = int(spec["prbs"])

env._invalidate_metric_caches()

ue_rows = []
for ue in env.get_all_ues():
    ue_rows.append({
        "ue_id": int(ue.id),
        "gnb_id": int(ue.serving_gnb),
        "slice_type": env.normalize_slice_type(ue.slice_type),
        "prbs": int(ue.prbs),
        "x": float(ue.x),
        "y": float(ue.y),
    })

ue_rows

## Compute Global Slice Loads

In [ ]:
slice_types = env._configured_slice_types()
gnb_ids = [int(gnb.id) for gnb in env.gnbs]

load_matrix = np.array([
    [env.estimate_slice_load(gnb_id, slice_type) for slice_type in slice_types]
    for gnb_id in gnb_ids
], dtype=float)

used_matrix = np.array([
    [env.get_slice_used_prbs(gnb_id, slice_type) for slice_type in slice_types]
    for gnb_id in gnb_ids
], dtype=int)

budget_matrix = np.array([
    [env.get_slice_prb_budget(gnb_id, slice_type) for slice_type in slice_types]
    for gnb_id in gnb_ids
], dtype=int)

print("Slice order:", slice_types)
print("gNB order:", gnb_ids)
print("Used PRBs:\n", used_matrix)
print("Slice PRB budgets:\n", budget_matrix)
print("Load L_i,s:\n", load_matrix)
print("Global observation with UE counts:", env.get_global_agent_observation(include_ue_counts=True))

## Plot Load Per Slice In Each gNodeB

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
image = ax.imshow(load_matrix, vmin=0.0, vmax=1.0, cmap="viridis")

ax.set_xticks(np.arange(len(slice_types)))
ax.set_xticklabels(slice_types)
ax.set_yticks(np.arange(len(gnb_ids)))
ax.set_yticklabels([f"gNB {gnb_id}" for gnb_id in gnb_ids])
ax.set_xlabel("Slice")
ax.set_ylabel("gNodeB")
ax.set_title("Upper-Agent PRB Slice Load $L_{i,s}$")

for row in range(load_matrix.shape[0]):
    for col in range(load_matrix.shape[1]):
        label = f"{load_matrix[row, col]:.2f}\n{used_matrix[row, col]}/{budget_matrix[row, col]} PRBs"
        ax.text(col, row, label, ha="center", va="center", color="white", fontsize=9)

cbar = fig.colorbar(image, ax=ax)
cbar.set_label("Clipped PRB load")
fig.tight_layout()
plt.show()

In [ ]:
x = np.arange(len(gnb_ids))
width = 0.24

fig, ax = plt.subplots(figsize=(7.5, 4.0))
for idx, slice_type in enumerate(slice_types):
    ax.bar(x + (idx - 1) * width, load_matrix[:, idx], width, label=slice_type)

ax.set_xticks(x)
ax.set_xticklabels([f"gNB {gnb_id}" for gnb_id in gnb_ids])
ax.set_ylim(0.0, 1.05)
ax.set_ylabel("Load")
ax.set_title("PRB-Based Load By gNodeB And Slice")
ax.legend(title="Slice")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
env.close()